> **Note**: This notebook has been upgraded to reflect the Phase 12 Final ML Pipeline Upgrade, utilizing the expanded 18-class taxonomy, ONNX export workflows, and accurate Boolean-mask object density calculation.

# 🚀 PlasticSense AI — Inference & Deployment (Notebook 08)

### 🌟 Overview
This notebook builds a **production-ready inference pipeline** for the trained YOLOv11 plastic detection model. It generates **reusable Python modules** that can be directly integrated into the FastAPI backend for real-time plastic detection.

### 📥 Inputs
| Asset | Path |
|---|---|
| Trained Model | `PlasticSense_AI/models/best.pt` |
| Test/Custom Images | Upload or specify folder |

### 📤 Outputs
All inference results are saved under `PlasticSense_AI/inference/`:
```
inference/
├── images/        # Annotated prediction images
├── json/          # Per-image structured JSON results
├── csv/           # Tabular detection results
└── reports/       # Batch summary reports
```

Additionally, production-ready backend modules are generated:
```
backend_ready/
├── detector.py        # Core detection class with detect() API
├── utils.py           # Drawing, I/O, and validation utilities
├── config.py          # Centralized configuration
└── requirements.txt   # Production dependencies
```

### ⚠️ Prerequisites
Run notebooks **01–07** first. This notebook does **NOT** retrain the model.

---
## 1. Environment Setup & Library Installation
Install and import all required dependencies for the inference pipeline.

In [ ]:
!pip install -q ultralytics rich pyyaml pandas opencv-python matplotlib pillow tqdm

In [ ]:
# ──────────────────────────────────────────────────────────
# Standard Library
# ──────────────────────────────────────────────────────────
import os
import sys
import json
import csv
import time
import shutil
import logging
import datetime
import warnings
import textwrap
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional, Union
from collections import defaultdict, Counter

# ──────────────────────────────────────────────────────────
# Third-Party
# ──────────────────────────────────────────────────────────
import torch
import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.progress import track

from ultralytics import YOLO

# ──────────────────────────────────────────────────────────
# Configuration
# ──────────────────────────────────────────────────────────
warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.figsize': (12, 8),
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1
})

console = Console()
SEED = 42
np.random.seed(SEED)

console.print('[bold green]✔ All libraries imported successfully.[/bold green]')

---
## 2. Logging System Initialization
Set up a dual-output logger (console + file) consistent with prior notebooks.

In [ ]:
def setup_logger(log_dir: Path, name: str = 'PlasticSense_Inference') -> logging.Logger:
    """Create a logger with file and console handlers."""
    log_dir.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    logger.handlers = []  # Reset

    fmt = logging.Formatter(
        '[%(asctime)s] %(levelname)s — %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )

    # File handler
    fh = logging.FileHandler(log_dir / 'inference.log', mode='w')
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    # Console handler
    ch = logging.StreamHandler(sys.stdout)
    ch.setFormatter(fmt)
    logger.addHandler(ch)

    return logger

---
## 3. Hardware Verification
Detect available compute accelerator (CUDA / MPS / CPU).

In [ ]:
def check_hardware() -> str:
    """Detect hardware accelerator and display status table."""
    table = Table(title='Hardware & Environment Status', show_header=True)
    table.add_column('Component', style='cyan')
    table.add_column('Status / Version', justify='right')

    table.add_row('Python Version', sys.version.split()[0])
    table.add_row('PyTorch Version', torch.__version__)

    device_type = 'cpu'
    if torch.cuda.is_available():
        device_name = torch.cuda.get_device_name(0)
        table.add_row('GPU Accelerator', f'[green]✔ {device_name}[/green]')
        table.add_row('CUDA Version', str(torch.version.cuda))
        device_type = 'cuda:0'
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        table.add_row('GPU Accelerator', '[green]✔ Apple Silicon (MPS)[/green]')
        device_type = 'mps'
    else:
        table.add_row('GPU Accelerator', '[bold red]✖ CPU ONLY[/bold red]')
        console.print('[bold yellow]⚠ GPU unavailable — inference will use CPU (slower).[/bold yellow]')

    import ultralytics
    table.add_row('Ultralytics Version', ultralytics.__version__)

    console.print(table)
    return device_type

DEVICE = check_hardware()

---
## 4. Project Paths & Directory Structure
Define all paths and create the inference output directory tree.

In [ ]:
# ──────────────────────────────────────────────────────────
# Detect Environment: Colab vs Local
# ──────────────────────────────────────────────────────────
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/PlasticSense_AI')
else:
    PROJECT_ROOT = Path('/Users/siddhivinayak/project/PlasticSense AI/Ml-model')

# ──────────────────────────────────────────────────────────
# Input Paths
# ──────────────────────────────────────────────────────────
MODELS_DIR     = PROJECT_ROOT / 'models'
BEST_PT_PATH   = MODELS_DIR / 'best.pt'
DATASET_DIR    = PROJECT_ROOT / 'Datasets' / 'augmented_yolo'
DATASET_YAML   = DATASET_DIR / 'dataset.yaml'
TEST_IMAGES    = DATASET_DIR / 'images' / 'test'

# ──────────────────────────────────────────────────────────
# Output Paths — Inference
# ──────────────────────────────────────────────────────────
INFERENCE_DIR  = PROJECT_ROOT / 'inference'
INF_IMAGES_DIR = INFERENCE_DIR / 'images'
INF_JSON_DIR   = INFERENCE_DIR / 'json'
INF_CSV_DIR    = INFERENCE_DIR / 'csv'
INF_REPORTS_DIR = INFERENCE_DIR / 'reports'
INF_LOGS_DIR   = INFERENCE_DIR / 'logs'

# ──────────────────────────────────────────────────────────
# Output Paths — Backend Modules
# ──────────────────────────────────────────────────────────
BACKEND_DIR    = PROJECT_ROOT / 'backend_ready'

# Create all output directories
for d in [INFERENCE_DIR, INF_IMAGES_DIR, INF_JSON_DIR,
          INF_CSV_DIR, INF_REPORTS_DIR, INF_LOGS_DIR, BACKEND_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Initialize logger
logger = setup_logger(INF_LOGS_DIR)
logger.info('Inference pipeline initialized.')

# Supported image extensions
SUPPORTED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}

console.print(Panel.fit(
    f'[bold cyan]Project Root:[/bold cyan]    {PROJECT_ROOT}\n'
    f'[bold cyan]Model Path:[/bold cyan]      {BEST_PT_PATH}\n'
    f'[bold cyan]Inference Dir:[/bold cyan]   {INFERENCE_DIR}\n'
    f'[bold cyan]Backend Dir:[/bold cyan]     {BACKEND_DIR}',
    title='📂 Project Configuration'
))

---
## 5. Model Loading & Verification
Load the trained `best.pt` model **once** and verify all properties. The model is NOT retrained.

In [ ]:
def load_model(model_path: Path, device: str = 'cpu') -> YOLO:
    """Load a trained YOLO model from checkpoint.

    Args:
        model_path: Path to the .pt model weights file.
        device: Compute device string ('cuda:0', 'mps', 'cpu').

    Returns:
        Loaded YOLO model ready for inference.

    Raises:
        FileNotFoundError: If the model file does not exist.
        RuntimeError: If the model fails to load.
    """
    # ── Verify model exists ──
    if not model_path.exists():
        # Attempt fallback from train_logs
        fallback = model_path.parent / 'train_logs' / 'PlasticSense_YOLOv11' / 'weights' / 'best.pt'
        if fallback.exists():
            shutil.copy2(fallback, model_path)
            console.print(f'[yellow]⚠ Copied best.pt from train_logs to {model_path}[/yellow]')
        else:
            raise FileNotFoundError(
                f'Model not found at {model_path}. '
                f'Run Notebook 06 (Training) first.'
            )

    console.print(f'[cyan]Loading model from {model_path}...[/cyan]')

    try:
        model = YOLO(str(model_path))
    except Exception as e:
        raise RuntimeError(f'Failed to load model: {e}')

    # ── Extract model metadata ──
    model_size_mb = model_path.stat().st_size / (1024 * 1024)
    class_names = model.names  # Dict[int, str]
    num_classes = len(class_names)
    total_params = sum(p.numel() for p in model.model.parameters())

    # ── Display verification table ──
    table = Table(title='🧠 Model Verification', show_header=True)
    table.add_column('Property', style='cyan', min_width=22)
    table.add_column('Value', justify='right', style='bold')

    table.add_row('Model Path', str(model_path.name))
    table.add_row('Model Size', f'{model_size_mb:.2f} MB')
    table.add_row('Number of Classes', str(num_classes))
    table.add_row('Class Names', ', '.join(class_names.values()))
    table.add_row('Total Parameters', f'{total_params:,}')
    table.add_row('Device', device)
    table.add_row('Status', '[bold green]✔ Loaded Successfully[/bold green]')

    console.print(table)
    logger.info(f'Model loaded: {model_path.name}, {num_classes} classes, {model_size_mb:.2f} MB')

    return model


# ── Load model ONCE ──
model = load_model(BEST_PT_PATH, DEVICE)
CLASS_NAMES: Dict[int, str] = model.names
NUM_CLASSES: int = len(CLASS_NAMES)

---
## 6. Color Palette for Visualization
Define a distinct, visually appealing color for each plastic class.

In [ ]:
# ──────────────────────────────────────────────────────────
# Class Color Palette (BGR for OpenCV)
# ──────────────────────────────────────────────────────────
CLASS_COLORS_BGR: Dict[int, Tuple[int, int, int]] = {
    0: (0, 165, 255),    # plastic_bottle  → Orange
    1: (147, 20, 255),   # plastic_bag     → Pink
    2: (0, 255, 127),    # wrapper         → Spring Green
    3: (255, 191, 0),    # styrofoam       → Deep Sky Blue
    4: (0, 255, 255),    # plastic_cap     → Yellow
    5: (255, 0, 0),      # food_container  → Blue
    6: (180, 105, 255),  # multilayer_pkg  → Hot Pink
    7: (0, 215, 255),    # other_plastic   → Gold
}

def get_class_color(class_id: int) -> Tuple[int, int, int]:
    """Get the BGR color for a given class ID."""
    return CLASS_COLORS_BGR.get(class_id, (128, 128, 128))

console.print('[green]✔ Color palette initialized for 8 classes.[/green]')

---
## 7. Core Inference Functions (Reusable)
These functions are designed for direct reuse in the FastAPI backend.

In [ ]:
def validate_image(image_path: Path) -> bool:
    """Validate that an image file exists, is supported, and is not corrupted.

    Args:
        image_path: Path to the image file.

    Returns:
        True if valid, False otherwise.
    """
    # Check existence
    if not image_path.exists():
        logger.warning(f'Image not found: {image_path}')
        return False

    # Check extension
    if image_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
        logger.warning(f'Unsupported format: {image_path.suffix} ({image_path.name})')
        return False

    # Check corruption
    try:
        img = cv2.imread(str(image_path))
        if img is None:
            logger.warning(f'Corrupted image (OpenCV failed): {image_path.name}')
            return False
        # Double-check with PIL
        pil_img = Image.open(image_path)
        pil_img.verify()
    except Exception as e:
        logger.warning(f'Corrupted image: {image_path.name} — {e}')
        return False

    return True


def predict_image(
    model: YOLO,
    image_path: Union[str, Path],
    conf_threshold: float = 0.25,
    iou_threshold: float = 0.5
) -> Dict[str, Any]:
    """Run YOLO inference on a single image and return structured results.

    Args:
        model: Loaded YOLO model.
        image_path: Path to the input image.
        conf_threshold: Minimum confidence score for detections.
        iou_threshold: IoU threshold for NMS.

    Returns:
        Dictionary containing detections, summary, and timing info.
    """
    image_path = Path(image_path)

    # ── Validate ──
    if not validate_image(image_path):
        return {
            'image_name': image_path.name,
            'image_size': 'N/A',
            'status': 'error',
            'error': 'Invalid or corrupted image',
            'detections': [],
            'summary': {
                'total_objects': 0,
                'plastic_types': 0,
                'average_confidence': 0.0
            }
        }

    # ── Read image dimensions ──
    img = cv2.imread(str(image_path))
    h, w = img.shape[:2]
    image_size = f'{w}x{h}'

    # ── Run inference ──
    start_time = time.perf_counter()
    results = model.predict(
        source=str(image_path),
        conf=conf_threshold,
        iou=iou_threshold,
        verbose=False
    )
    inference_time = (time.perf_counter() - start_time) * 1000  # ms

    # ── Extract detections ──
    detections: List[Dict[str, Any]] = []
    class_counter: Counter = Counter()
    confidences: List[float] = []

    if len(results) > 0 and results[0].boxes is not None:
        boxes = results[0].boxes
        for i, box in enumerate(boxes):
            cls_id = int(box.cls.item())
            conf = float(box.conf.item())
            xyxy = box.xyxy[0].cpu().numpy().tolist()
            # Convert xyxy to [x, y, w, h] format
            x1, y1, x2, y2 = xyxy
            bbox_xywh = [
                round(x1, 2),
                round(y1, 2),
                round(x2 - x1, 2),
                round(y2 - y1, 2)
            ]

            class_name = model.names.get(cls_id, f'class_{cls_id}')
            class_counter[class_name] += 1
            confidences.append(conf)

            detections.append({
                'id': i + 1,
                'class': class_name,
                'class_id': cls_id,
                'confidence': round(conf, 4),
                'bbox': bbox_xywh
            })

    # ── Build summary ──
    total_objects = len(detections)
    plastic_types = len(class_counter)
    avg_confidence = round(float(np.mean(confidences)), 4) if confidences else 0.0
    dominant_type = class_counter.most_common(1)[0][0] if class_counter else 'N/A'

    result = {
        'image_name': image_path.name,
        'image_size': image_size,
        'status': 'success',
        'detections': detections,
        'summary': {
            'total_objects': total_objects,
            'plastic_types': plastic_types,
            'average_confidence': avg_confidence,
            'dominant_type': dominant_type,
            'objects_per_class': dict(class_counter),
            'inference_time_ms': round(inference_time, 2),
            'detection_time_ms': round(inference_time, 2)
        }
    }

    logger.info(
        f'Predicted {image_path.name}: '
        f'{total_objects} objects, {plastic_types} types, '
        f'{avg_confidence:.4f} avg conf, {inference_time:.1f} ms'
    )

    return result


console.print('[bold green]✔ Core inference function defined: predict_image()[/bold green]')

---
## 8. Visualization — Draw Bounding Boxes
Draw annotated bounding boxes with class names, confidence scores, and object IDs on images.

In [ ]:
def draw_boxes(
    image_path: Union[str, Path],
    detections: List[Dict[str, Any]],
    output_path: Optional[Path] = None,
    show: bool = False
) -> np.ndarray:
    """Draw bounding boxes with class names, confidence, and object ID.

    Args:
        image_path: Path to the original image.
        detections: List of detection dicts from predict_image().
        output_path: Optional path to save the annotated image.
        show: Whether to display inline (for notebooks).

    Returns:
        Annotated image as numpy array (BGR).
    """
    image_path = Path(image_path)
    img = cv2.imread(str(image_path))
    if img is None:
        logger.error(f'Cannot read image for drawing: {image_path}')
        return np.zeros((100, 100, 3), dtype=np.uint8)

    h, w = img.shape[:2]
    # Scale font/line thickness based on image size
    scale = min(w, h) / 640
    thickness = max(int(2 * scale), 1)
    font_scale = max(0.5 * scale, 0.4)

    for det in detections:
        cls_id = det.get('class_id', 0)
        cls_name = det['class']
        conf = det['confidence']
        obj_id = det.get('id', 0)
        x, y, bw, bh = det['bbox']

        # Convert [x, y, w, h] back to [x1, y1, x2, y2]
        x1, y1 = int(x), int(y)
        x2, y2 = int(x + bw), int(y + bh)

        color = get_class_color(cls_id)

        # Draw bounding box
        cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)

        # Label text
        label = f'#{obj_id} {cls_name} {conf:.2f}'
        (tw, th), baseline = cv2.getTextSize(
            label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness
        )

        # Label background
        label_y1 = max(y1 - th - baseline - 6, 0)
        cv2.rectangle(img, (x1, label_y1), (x1 + tw + 4, y1), color, -1)

        # Label text (white on colored background)
        cv2.putText(
            img, label, (x1 + 2, y1 - baseline - 2),
            cv2.FONT_HERSHEY_SIMPLEX, font_scale, (255, 255, 255),
            max(thickness - 1, 1), cv2.LINE_AA
        )

    # ── Save annotated image ──
    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(output_path), img)
        logger.info(f'Annotated image saved: {output_path.name}')

    # ── Display inline ──
    if show:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(14, 10))
        plt.imshow(img_rgb)
        plt.title(
            f'{image_path.name} — {len(detections)} detections',
            fontweight='bold', fontsize=14
        )
        plt.axis('off')
        plt.tight_layout()
        plt.show()
        plt.close()

    return img


console.print('[bold green]✔ Visualization function defined: draw_boxes()[/bold green]')

---
## 9. Export Functions — JSON & CSV
Save detection results in structured JSON and tabular CSV formats.

In [ ]:
def save_json(
    result: Dict[str, Any],
    output_dir: Path
) -> Path:
    """Save a single image's detection result as JSON.

    Args:
        result: Detection result dict from predict_image().
        output_dir: Directory to save the JSON file.

    Returns:
        Path to the saved JSON file.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    image_stem = Path(result['image_name']).stem
    json_path = output_dir / f'{image_stem}.json'

    with open(json_path, 'w') as f:
        json.dump(result, f, indent=4, default=str)

    logger.info(f'JSON saved: {json_path.name}')
    return json_path


def save_csv(
    results: List[Dict[str, Any]],
    output_dir: Path,
    filename: str = 'detection_results.csv'
) -> Path:
    """Save all detection results as a flat CSV file.

    Args:
        results: List of detection result dicts.
        output_dir: Directory to save the CSV file.
        filename: Name of the CSV file.

    Returns:
        Path to the saved CSV file.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    csv_path = output_dir / filename

    rows: List[Dict[str, Any]] = []
    for res in results:
        if res.get('status') != 'success':
            rows.append({
                'image_name': res['image_name'],
                'image_size': res.get('image_size', 'N/A'),
                'status': res.get('status', 'error'),
                'object_id': '', 'class': '', 'class_id': '',
                'confidence': '', 'bbox_x': '', 'bbox_y': '',
                'bbox_w': '', 'bbox_h': '',
                'total_objects': 0, 'plastic_types': 0,
                'average_confidence': 0.0, 'dominant_type': 'N/A',
                'inference_time_ms': 0.0
            })
            continue

        summary = res['summary']

        if not res['detections']:
            rows.append({
                'image_name': res['image_name'],
                'image_size': res['image_size'],
                'status': 'success',
                'object_id': '', 'class': 'none', 'class_id': '',
                'confidence': '', 'bbox_x': '', 'bbox_y': '',
                'bbox_w': '', 'bbox_h': '',
                'total_objects': 0, 'plastic_types': 0,
                'average_confidence': 0.0,
                'dominant_type': 'N/A',
                'inference_time_ms': summary.get('inference_time_ms', 0)
            })
        else:
            for det in res['detections']:
                rows.append({
                    'image_name': res['image_name'],
                    'image_size': res['image_size'],
                    'status': 'success',
                    'object_id': det['id'],
                    'class': det['class'],
                    'class_id': det['class_id'],
                    'confidence': det['confidence'],
                    'bbox_x': det['bbox'][0],
                    'bbox_y': det['bbox'][1],
                    'bbox_w': det['bbox'][2],
                    'bbox_h': det['bbox'][3],
                    'total_objects': summary['total_objects'],
                    'plastic_types': summary['plastic_types'],
                    'average_confidence': summary['average_confidence'],
                    'dominant_type': summary.get('dominant_type', 'N/A'),
                    'inference_time_ms': summary.get('inference_time_ms', 0)
                })

    df = pd.DataFrame(rows)
    df.to_csv(csv_path, index=False)
    logger.info(f'CSV saved: {csv_path.name} ({len(rows)} rows)')
    return csv_path


console.print('[bold green]✔ Export functions defined: save_json(), save_csv()[/bold green]')

---
## 10. Batch Inference — predict_folder()
Run inference on an entire folder of images with progress tracking and statistics.

In [ ]:
def predict_folder(
    model: YOLO,
    folder_path: Union[str, Path],
    output_images_dir: Path,
    output_json_dir: Path,
    output_csv_dir: Path,
    output_reports_dir: Path,
    conf_threshold: float = 0.25,
    iou_threshold: float = 0.5,
    show_samples: int = 5
) -> Dict[str, Any]:
    """Run inference on all supported images in a folder.

    Args:
        model: Loaded YOLO model.
        folder_path: Path to folder containing images.
        output_images_dir: Dir to save annotated images.
        output_json_dir: Dir to save per-image JSON results.
        output_csv_dir: Dir to save CSV results.
        output_reports_dir: Dir to save batch summary report.
        conf_threshold: Minimum confidence score.
        iou_threshold: IoU threshold for NMS.
        show_samples: Number of sample predictions to display.

    Returns:
        Batch summary dict with aggregate statistics.
    """
    folder_path = Path(folder_path)
    if not folder_path.exists() or not folder_path.is_dir():
        raise FileNotFoundError(f'Folder not found: {folder_path}')

    # ── Discover images ──
    image_files = sorted([
        f for f in folder_path.iterdir()
        if f.is_file() and f.suffix.lower() in SUPPORTED_EXTENSIONS
    ])

    total_images = len(image_files)
    if total_images == 0:
        console.print(f'[bold red]✖ No supported images found in {folder_path}[/bold red]')
        return {'total_images': 0, 'processed': 0}

    console.print(f'[cyan]Found {total_images} images in {folder_path.name}/[/cyan]')

    # ── Process each image ──
    all_results: List[Dict[str, Any]] = []
    inference_times: List[float] = []
    total_plastic_count = 0
    global_class_counter: Counter = Counter()
    all_confidences: List[float] = []
    processed_count = 0
    error_count = 0

    batch_start = time.perf_counter()

    for img_path in tqdm(image_files, desc='🔍 Running Inference', unit='img'):
        # Predict
        result = predict_image(model, img_path, conf_threshold, iou_threshold)
        all_results.append(result)

        if result.get('status') != 'success':
            error_count += 1
            continue

        processed_count += 1
        summary = result['summary']

        # Accumulate stats
        inference_times.append(summary['inference_time_ms'])
        total_plastic_count += summary['total_objects']
        for cls_name, cnt in summary.get('objects_per_class', {}).items():
            global_class_counter[cls_name] += cnt
        for det in result['detections']:
            all_confidences.append(det['confidence'])

        # Draw and save annotated image
        annotated_path = output_images_dir / f'pred_{img_path.name}'
        draw_boxes(img_path, result['detections'], output_path=annotated_path)

        # Save per-image JSON
        save_json(result, output_json_dir)

    batch_elapsed = (time.perf_counter() - batch_start) * 1000  # ms

    # ── Save CSV for all results ──
    save_csv(all_results, output_csv_dir)

    # ── Compute batch statistics ──
    avg_inference_time = round(float(np.mean(inference_times)), 2) if inference_times else 0.0
    avg_fps = round(1000.0 / avg_inference_time, 2) if avg_inference_time > 0 else 0.0
    avg_confidence = round(float(np.mean(all_confidences)), 4) if all_confidences else 0.0
    dominant_overall = global_class_counter.most_common(1)[0][0] if global_class_counter else 'N/A'

    batch_summary = {
        'project': 'PlasticSense AI',
        'notebook': '08_Inference_and_Deployment',
        'timestamp': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'source_folder': str(folder_path),
        'total_images': total_images,
        'processed_images': processed_count,
        'error_images': error_count,
        'total_plastic_objects': total_plastic_count,
        'unique_plastic_types': len(global_class_counter),
        'dominant_plastic_type': dominant_overall,
        'objects_per_class': dict(global_class_counter),
        'average_confidence': avg_confidence,
        'average_inference_time_ms': avg_inference_time,
        'average_fps': avg_fps,
        'total_batch_time_ms': round(batch_elapsed, 2),
        'conf_threshold': conf_threshold,
        'iou_threshold': iou_threshold
    }

    # ── Save batch summary report ──
    report_path = output_reports_dir / 'batch_inference_report.json'
    with open(report_path, 'w') as f:
        json.dump(batch_summary, f, indent=4, default=str)

    report_csv_path = output_reports_dir / 'batch_inference_report.csv'
    pd.DataFrame([batch_summary]).to_csv(report_csv_path, index=False)

    # ── Detection summary CSV ──
    detection_summary_rows = []
    for res in all_results:
        if res.get('status') == 'success':
            s = res['summary']
            detection_summary_rows.append({
                'image_name': res['image_name'],
                'total_objects': s['total_objects'],
                'plastic_types': s['plastic_types'],
                'average_confidence': s['average_confidence'],
                'dominant_type': s.get('dominant_type', 'N/A'),
                'inference_time_ms': s.get('inference_time_ms', 0)
            })
    if detection_summary_rows:
        df_summary = pd.DataFrame(detection_summary_rows)
        df_summary.to_csv(output_reports_dir / 'detection_summary.csv', index=False)

    # ── Display batch results table ──
    table = Table(
        title='📊 Batch Inference Results',
        show_header=True, header_style='bold magenta'
    )
    table.add_column('Metric', style='cyan', min_width=28)
    table.add_column('Value', justify='right', style='bold')

    table.add_row('Total Images', str(total_images))
    table.add_row('Processed Images', f'[green]{processed_count}[/green]')
    table.add_row('Error Images', f'[red]{error_count}[/red]' if error_count else '0')
    table.add_row('Total Plastic Objects', str(total_plastic_count))
    table.add_row('Unique Plastic Types', str(len(global_class_counter)))
    table.add_row('Dominant Type', dominant_overall)
    table.add_row('Average Confidence', f'{avg_confidence:.4f}')
    table.add_row('Average Inference Time', f'{avg_inference_time:.2f} ms')
    table.add_row('Average FPS', f'{avg_fps:.2f}')
    table.add_row('Total Batch Time', f'{batch_elapsed/1000:.2f} s')

    console.print(table)

    # ── Per-class breakdown ──
    if global_class_counter:
        cls_table = Table(
            title='📋 Objects Per Class',
            show_header=True, header_style='bold green'
        )
        cls_table.add_column('Class', style='cyan')
        cls_table.add_column('Count', justify='right', style='bold')
        cls_table.add_column('Percentage', justify='right')

        for cls_name, count in global_class_counter.most_common():
            pct = (count / total_plastic_count * 100) if total_plastic_count > 0 else 0
            cls_table.add_row(cls_name, str(count), f'{pct:.1f}%')

        console.print(cls_table)

    # ── Show sample predictions ──
    sample_results = [
        r for r in all_results
        if r.get('status') == 'success' and r['summary']['total_objects'] > 0
    ][:show_samples]

    if sample_results:
        console.print(f'\n[bold cyan]─── Sample Predictions ({len(sample_results)} shown) ───[/bold cyan]')
        for res in sample_results:
            img_path = folder_path / res['image_name']
            draw_boxes(img_path, res['detections'], show=True)

    logger.info(
        f'Batch complete: {processed_count}/{total_images} images, '
        f'{total_plastic_count} objects, {avg_fps:.1f} FPS'
    )

    return batch_summary


console.print('[bold green]✔ Batch inference function defined: predict_folder()[/bold green]')

---
## 11. Run Inference on Test Dataset
Execute the full inference pipeline on the test split to validate the pipeline end-to-end.

In [ ]:
# ── Verify test images exist ──
if TEST_IMAGES.exists():
    test_image_count = len([
        f for f in TEST_IMAGES.iterdir()
        if f.suffix.lower() in SUPPORTED_EXTENSIONS
    ])
    console.print(f'[cyan]Test folder: {TEST_IMAGES} ({test_image_count} images)[/cyan]')

    batch_results = predict_folder(
        model=model,
        folder_path=TEST_IMAGES,
        output_images_dir=INF_IMAGES_DIR,
        output_json_dir=INF_JSON_DIR,
        output_csv_dir=INF_CSV_DIR,
        output_reports_dir=INF_REPORTS_DIR,
        conf_threshold=0.25,
        iou_threshold=0.5,
        show_samples=5
    )
else:
    console.print('[bold red]✖ Test images directory not found. Skipping test inference.[/bold red]')
    console.print(f'[yellow]Expected: {TEST_IMAGES}[/yellow]')
    batch_results = {'total_images': 0, 'processed_images': 0}

---
## 12. Single Image Inference Demo
Demonstrate inference on a single image with full output display.

In [ ]:
def demo_single_image(model: YOLO, images_dir: Path) -> None:
    """Run and display inference on a single sample image."""
    # Find a sample image
    sample_images = sorted([
        f for f in images_dir.iterdir()
        if f.suffix.lower() in SUPPORTED_EXTENSIONS
    ])

    if not sample_images:
        console.print('[yellow]⚠ No sample images available for demo.[/yellow]')
        return

    sample_path = sample_images[0]
    console.print(f'\n[bold cyan]─── Single Image Demo: {sample_path.name} ───[/bold cyan]')

    # Run prediction
    result = predict_image(model, sample_path)

    # Display JSON output
    console.print('\n[bold]📄 JSON Output:[/bold]')
    console.print_json(json.dumps(result, indent=2, default=str))

    # Display annotated image
    if result['status'] == 'success':
        draw_boxes(sample_path, result['detections'], show=True)


if TEST_IMAGES.exists():
    demo_single_image(model, TEST_IMAGES)

---
## 13. Drag-and-Drop Upload Support (Colab)
Upload custom images directly from your computer for inference.

In [ ]:
def run_upload_inference(model: YOLO) -> List[Dict[str, Any]]:
    """Handle drag-and-drop image upload and run inference.

    Only works in Google Colab environment.

    Returns:
        List of detection result dicts for uploaded images.
    """
    if not IS_COLAB:
        console.print('[yellow]⚠ Upload is only supported in Google Colab.[/yellow]')
        console.print('[cyan]Use predict_image() or predict_folder() for local inference.[/cyan]')
        return []

    from google.colab import files

    console.print('[bold cyan]📤 Upload images for inference (JPG, JPEG, PNG, WEBP):[/bold cyan]')
    uploaded = files.upload()

    if not uploaded:
        console.print('[yellow]No files uploaded.[/yellow]')
        return []

    upload_dir = INFERENCE_DIR / 'uploaded'
    upload_dir.mkdir(parents=True, exist_ok=True)

    results = []
    for filename, data in uploaded.items():
        file_path = upload_dir / filename
        with open(file_path, 'wb') as f:
            f.write(data)

        # Check extension
        if file_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            console.print(f'[red]✖ Unsupported format: {filename}[/red]')
            continue

        # Predict
        result = predict_image(model, file_path)
        results.append(result)

        # Save annotated image
        if result['status'] == 'success':
            annotated_path = INF_IMAGES_DIR / f'upload_pred_{filename}'
            draw_boxes(file_path, result['detections'],
                       output_path=annotated_path, show=True)

            # Save JSON
            save_json(result, INF_JSON_DIR)

            # Print summary
            s = result['summary']
            console.print(
                f'[green]✔ {filename}: {s["total_objects"]} objects, '
                f'{s["plastic_types"]} types, '
                f'{s["average_confidence"]:.4f} avg conf[/green]'
            )

    if results:
        save_csv(results, INF_CSV_DIR, filename='upload_results.csv')
        console.print(f'[bold green]✔ Processed {len(results)} uploaded images.[/bold green]')

    return results


# Uncomment the line below to enable upload inference:
# upload_results = run_upload_inference(model)

---
## 14. Google Drive Folder Inference
Run inference on images from a custom Google Drive folder.

In [ ]:
def run_gdrive_inference(
    model: YOLO,
    gdrive_folder: str = '/content/drive/MyDrive/custom_images'
) -> Dict[str, Any]:
    """Run inference on images from a Google Drive folder.

    Args:
        model: Loaded YOLO model.
        gdrive_folder: Path to Google Drive folder with images.

    Returns:
        Batch summary dict.
    """
    folder = Path(gdrive_folder)

    if not folder.exists():
        console.print(f'[yellow]⚠ Google Drive folder not found: {folder}[/yellow]')
        console.print('[cyan]Ensure Google Drive is mounted and the path is correct.[/cyan]')
        return {'total_images': 0}

    console.print(f'[cyan]Running inference on Google Drive folder: {folder}[/cyan]')

    # Create separate output dirs for GDrive inference
    gdrive_out = INFERENCE_DIR / 'gdrive_results'
    gdrive_images = gdrive_out / 'images'
    gdrive_json = gdrive_out / 'json'
    gdrive_csv = gdrive_out / 'csv'
    gdrive_reports = gdrive_out / 'reports'

    for d in [gdrive_images, gdrive_json, gdrive_csv, gdrive_reports]:
        d.mkdir(parents=True, exist_ok=True)

    return predict_folder(
        model=model,
        folder_path=folder,
        output_images_dir=gdrive_images,
        output_json_dir=gdrive_json,
        output_csv_dir=gdrive_csv,
        output_reports_dir=gdrive_reports
    )


# Uncomment and set your Google Drive folder path:
# gdrive_results = run_gdrive_inference(model, '/content/drive/MyDrive/my_plastic_images')

---
## 15. Verify Inference Outputs
List and verify all generated output files.

In [ ]:
def verify_outputs(output_dir: Path) -> None:
    """List and verify all generated inference outputs."""
    table = Table(
        title='📦 Inference Outputs',
        show_header=True, header_style='bold cyan'
    )
    table.add_column('Directory', style='cyan', min_width=16)
    table.add_column('File', min_width=36)
    table.add_column('Size', justify='right')

    total_files = 0
    for subdir in sorted(output_dir.rglob('*')):
        if subdir.is_file():
            rel_dir = subdir.parent.relative_to(output_dir)
            size_kb = subdir.stat().st_size / 1024
            size_str = f'{size_kb:.1f} KB' if size_kb < 1024 else f'{size_kb/1024:.1f} MB'
            table.add_row(str(rel_dir), subdir.name, size_str)
            total_files += 1

    console.print(table)
    console.print(f'\n[bold]Total files generated: {total_files}[/bold]')


verify_outputs(INFERENCE_DIR)

---
## 16. Generate Backend-Ready Python Modules
Automatically create production-ready Python files that can be directly copied into the FastAPI backend.

In [ ]:
# ════════════════════════════════════════════════════════════
# 16a. Generate config.py
# ════════════════════════════════════════════════════════════

config_py_content = textwrap.dedent('''\
"""
PlasticSense AI — Backend Configuration
Auto-generated by Notebook 08: Inference & Deployment.
Do NOT edit manually unless you know what you are doing.
"""
from pathlib import Path
from typing import Dict, Tuple

# ──────────────────────────────────────────────────────────
# Model Configuration
# ──────────────────────────────────────────────────────────
MODEL_PATH: str = "models/best.pt"
CONFIDENCE_THRESHOLD: float = 0.25
IOU_THRESHOLD: float = 0.5
IMAGE_SIZE: int = 640

# ──────────────────────────────────────────────────────────
# Supported Image Formats
# ──────────────────────────────────────────────────────────
SUPPORTED_EXTENSIONS: set = {".jpg", ".jpeg", ".png", ".webp"}

# ──────────────────────────────────────────────────────────
# Class Names
# ──────────────────────────────────────────────────────────
CLASS_NAMES: Dict[int, str] = {
    0: "plastic_bottle",
    1: "plastic_bag",
    2: "wrapper",
    3: "styrofoam",
    4: "plastic_cap",
    5: "food_container",
    6: "multilayer_packaging",
    7: "other_plastic",
}

NUM_CLASSES: int = len(CLASS_NAMES)

# ──────────────────────────────────────────────────────────
# Visualization Colors (BGR for OpenCV)
# ──────────────────────────────────────────────────────────
CLASS_COLORS_BGR: Dict[int, Tuple[int, int, int]] = {
    0: (0, 165, 255),
    1: (147, 20, 255),
    2: (0, 255, 127),
    3: (255, 191, 0),
    4: (0, 255, 255),
    5: (255, 0, 0),
    6: (180, 105, 255),
    7: (0, 215, 255),
}

# ──────────────────────────────────────────────────────────
# Output Directories
# ──────────────────────────────────────────────────────────
OUTPUT_DIR: str = "inference_output"
IMAGES_DIR: str = f"{OUTPUT_DIR}/images"
JSON_DIR: str = f"{OUTPUT_DIR}/json"
CSV_DIR: str = f"{OUTPUT_DIR}/csv"
''')

config_path = BACKEND_DIR / 'config.py'
with open(config_path, 'w') as f:
    f.write(config_py_content)

console.print(f'[green]✔ Generated: {config_path}[/green]')
logger.info(f'Backend module generated: config.py')

In [ ]:
# ════════════════════════════════════════════════════════════
# 16b. Generate utils.py
# ════════════════════════════════════════════════════════════

utils_py_content = textwrap.dedent('''\
"""
PlasticSense AI — Utility Functions
Auto-generated by Notebook 08: Inference & Deployment.
Provides image validation, drawing, and I/O helpers.
"""
import json
import logging
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple, Union

import cv2
import numpy as np
import pandas as pd
from PIL import Image

from config import SUPPORTED_EXTENSIONS, CLASS_COLORS_BGR

logger = logging.getLogger("PlasticSense_AI")


def validate_image(image_path: Path) -> bool:
    """Validate that an image file exists, is supported, and is not corrupted.

    Args:
        image_path: Path to the image file.

    Returns:
        True if valid, False otherwise.
    """
    if not image_path.exists():
        logger.warning(f"Image not found: {image_path}")
        return False

    if image_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
        logger.warning(f"Unsupported format: {image_path.suffix} ({image_path.name})")
        return False

    try:
        img = cv2.imread(str(image_path))
        if img is None:
            logger.warning(f"Corrupted image (OpenCV): {image_path.name}")
            return False
        pil_img = Image.open(image_path)
        pil_img.verify()
    except Exception as e:
        logger.warning(f"Corrupted image: {image_path.name} — {e}")
        return False

    return True


def get_class_color(class_id: int) -> Tuple[int, int, int]:
    """Get the BGR color for a given class ID."""
    return CLASS_COLORS_BGR.get(class_id, (128, 128, 128))


def draw_boxes(
    image_path: Union[str, Path],
    detections: List[Dict[str, Any]],
    output_path: Optional[Path] = None
) -> np.ndarray:
    """Draw bounding boxes with class names, confidence, and object ID.

    Args:
        image_path: Path to the original image.
        detections: List of detection dicts.
        output_path: Optional path to save the annotated image.

    Returns:
        Annotated image as numpy array (BGR).
    """
    image_path = Path(image_path)
    img = cv2.imread(str(image_path))
    if img is None:
        logger.error(f"Cannot read image: {image_path}")
        return np.zeros((100, 100, 3), dtype=np.uint8)

    h, w = img.shape[:2]
    scale = min(w, h) / 640
    thickness = max(int(2 * scale), 1)
    font_scale = max(0.5 * scale, 0.4)

    for det in detections:
        cls_id = det.get("class_id", 0)
        cls_name = det["class"]
        conf = det["confidence"]
        obj_id = det.get("id", 0)
        x, y, bw, bh = det["bbox"]

        x1, y1 = int(x), int(y)
        x2, y2 = int(x + bw), int(y + bh)
        color = get_class_color(cls_id)

        cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)

        label = f"#{obj_id} {cls_name} {conf:.2f}"
        (tw, th_text), baseline = cv2.getTextSize(
            label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness
        )

        label_y1 = max(y1 - th_text - baseline - 6, 0)
        cv2.rectangle(img, (x1, label_y1), (x1 + tw + 4, y1), color, -1)
        cv2.putText(
            img, label, (x1 + 2, y1 - baseline - 2),
            cv2.FONT_HERSHEY_SIMPLEX, font_scale, (255, 255, 255),
            max(thickness - 1, 1), cv2.LINE_AA
        )

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(output_path), img)

    return img


def save_json(result: Dict[str, Any], output_dir: Path) -> Path:
    """Save detection result as JSON."""
    output_dir.mkdir(parents=True, exist_ok=True)
    image_stem = Path(result["image_name"]).stem
    json_path = output_dir / f"{image_stem}.json"

    with open(json_path, "w") as f:
        json.dump(result, f, indent=4, default=str)

    return json_path


def save_csv(
    results: List[Dict[str, Any]],
    output_dir: Path,
    filename: str = "detection_results.csv"
) -> Path:
    """Save all detection results as a flat CSV file."""
    output_dir.mkdir(parents=True, exist_ok=True)
    csv_path = output_dir / filename

    rows = []
    for res in results:
        if res.get("status") != "success" or not res["detections"]:
            rows.append({
                "image_name": res["image_name"],
                "status": res.get("status", "error"),
                "class": "", "confidence": "",
                "bbox_x": "", "bbox_y": "",
                "bbox_w": "", "bbox_h": "",
                "total_objects": 0
            })
            continue

        for det in res["detections"]:
            rows.append({
                "image_name": res["image_name"],
                "status": "success",
                "class": det["class"],
                "confidence": det["confidence"],
                "bbox_x": det["bbox"][0],
                "bbox_y": det["bbox"][1],
                "bbox_w": det["bbox"][2],
                "bbox_h": det["bbox"][3],
                "total_objects": res["summary"]["total_objects"]
            })

    df = pd.DataFrame(rows)
    df.to_csv(csv_path, index=False)
    return csv_path
''')

utils_path = BACKEND_DIR / 'utils.py'
with open(utils_path, 'w') as f:
    f.write(utils_py_content)

console.print(f'[green]✔ Generated: {utils_path}[/green]')
logger.info(f'Backend module generated: utils.py')

In [ ]:
# ════════════════════════════════════════════════════════════
# 16c. Generate detector.py
# ════════════════════════════════════════════════════════════

detector_py_content = textwrap.dedent('''\
"""
PlasticSense AI — Detector Module
Auto-generated by Notebook 08: Inference & Deployment.

This module provides the core detection API for the FastAPI backend.
Usage:
    from detector import PlasticDetector
    detector = PlasticDetector()
    result = detector.detect("path/to/image.jpg")
"""
import time
import logging
from pathlib import Path
from typing import Dict, List, Any, Optional, Union
from collections import Counter

import cv2
import numpy as np
import torch
from ultralytics import YOLO

from config import (
    MODEL_PATH,
    CONFIDENCE_THRESHOLD,
    IOU_THRESHOLD,
    SUPPORTED_EXTENSIONS,
    CLASS_NAMES,
)
from utils import validate_image, draw_boxes, save_json

logger = logging.getLogger("PlasticSense_AI")


class PlasticDetector:
    """Production-ready plastic waste detector using YOLOv11.

    Attributes:
        model: Loaded YOLO model instance.
        device: Compute device string.
        class_names: Mapping from class ID to class name.
    """

    def __init__(
        self,
        model_path: str = MODEL_PATH,
        conf_threshold: float = CONFIDENCE_THRESHOLD,
        iou_threshold: float = IOU_THRESHOLD,
    ) -> None:
        """Initialize the detector with model weights.

        Args:
            model_path: Path to the .pt model weights.
            conf_threshold: Minimum confidence for detections.
            iou_threshold: IoU threshold for NMS.

        Raises:
            FileNotFoundError: If model weights are not found.
        """
        self.model_path = Path(model_path)
        self.conf_threshold = conf_threshold
        self.iou_threshold = iou_threshold

        if not self.model_path.exists():
            raise FileNotFoundError(f"Model not found: {self.model_path}")

        # Detect device
        if torch.cuda.is_available():
            self.device = "cuda:0"
        elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            self.device = "mps"
        else:
            self.device = "cpu"

        # Load model once
        self.model = YOLO(str(self.model_path))
        self.class_names = self.model.names
        logger.info(
            f"PlasticDetector initialized: {self.model_path.name}, "
            f"{len(self.class_names)} classes, device={self.device}"
        )

    def detect(self, image_path: Union[str, Path]) -> Dict[str, Any]:
        """Run plastic detection on a single image.

        Args:
            image_path: Path to the input image.

        Returns:
            Structured JSON-compatible dict with detections and summary.
        """
        image_path = Path(image_path)

        # Validate
        if not validate_image(image_path):
            return {
                "image_name": image_path.name,
                "image_size": "N/A",
                "status": "error",
                "error": "Invalid or corrupted image",
                "detections": [],
                "summary": {
                    "total_objects": 0,
                    "plastic_types": 0,
                    "average_confidence": 0.0,
                },
            }

        # Read image dimensions
        img = cv2.imread(str(image_path))
        h, w = img.shape[:2]

        # Inference
        start = time.perf_counter()
        results = self.model.predict(
            source=str(image_path),
            conf=self.conf_threshold,
            iou=self.iou_threshold,
            verbose=False,
        )
        inference_time = (time.perf_counter() - start) * 1000

        # Extract detections
        detections: List[Dict[str, Any]] = []
        class_counter: Counter = Counter()
        confidences: List[float] = []

        if len(results) > 0 and results[0].boxes is not None:
            for i, box in enumerate(results[0].boxes):
                cls_id = int(box.cls.item())
                conf = float(box.conf.item())
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().tolist()

                class_name = self.class_names.get(cls_id, f"class_{cls_id}")
                class_counter[class_name] += 1
                confidences.append(conf)

                detections.append({
                    "id": i + 1,
                    "class": class_name,
                    "class_id": cls_id,
                    "confidence": round(conf, 4),
                    "bbox": [
                        round(x1, 2), round(y1, 2),
                        round(x2 - x1, 2), round(y2 - y1, 2),
                    ],
                })

        total_objects = len(detections)
        avg_conf = round(float(np.mean(confidences)), 4) if confidences else 0.0
        dominant = class_counter.most_common(1)[0][0] if class_counter else "N/A"

        return {
            "image_name": image_path.name,
            "image_size": f"{w}x{h}",
            "status": "success",
            "detections": detections,
            "summary": {
                "total_objects": total_objects,
                "plastic_types": len(class_counter),
                "average_confidence": avg_conf,
                "dominant_type": dominant,
                "objects_per_class": dict(class_counter),
                "inference_time_ms": round(inference_time, 2),
                "detection_time_ms": round(inference_time, 2),
            },
        }


def detect(image_path: Union[str, Path]) -> Dict[str, Any]:
    """Convenience function for single-image detection.

    This function initializes the detector on first call
    and reuses it for subsequent calls (singleton pattern).

    Args:
        image_path: Path to the input image.

    Returns:
        Structured JSON-compatible detection result.
    """
    if not hasattr(detect, "_detector"):
        detect._detector = PlasticDetector()
    return detect._detector.detect(image_path)
''')

detector_path = BACKEND_DIR / 'detector.py'
with open(detector_path, 'w') as f:
    f.write(detector_py_content)

console.print(f'[green]✔ Generated: {detector_path}[/green]')
logger.info(f'Backend module generated: detector.py')

In [ ]:
# ════════════════════════════════════════════════════════════
# 16d. Generate requirements.txt
# ════════════════════════════════════════════════════════════

requirements_content = textwrap.dedent('''\
# PlasticSense AI — Production Dependencies
# Auto-generated by Notebook 08: Inference & Deployment.

# Core ML
ultralytics>=8.0.0
torch>=2.0.0
torchvision>=0.15.0

# Computer Vision
opencv-python-headless>=4.8.0
Pillow>=10.0.0

# Data Processing
numpy>=1.24.0
pandas>=2.0.0

# API Framework
fastapi>=0.100.0
uvicorn>=0.23.0
python-multipart>=0.0.6

# Utilities
pyyaml>=6.0
tqdm>=4.65.0
rich>=13.0.0
''')

req_path = BACKEND_DIR / 'requirements.txt'
with open(req_path, 'w') as f:
    f.write(requirements_content)

console.print(f'[green]✔ Generated: {req_path}[/green]')
logger.info(f'Backend module generated: requirements.txt')

---
## 17. Verify Backend Modules
List all generated backend-ready files and verify their integrity.

In [ ]:
def verify_backend_modules(backend_dir: Path) -> None:
    """Verify all generated backend modules exist and display summary."""
    expected_files = ['detector.py', 'utils.py', 'config.py', 'requirements.txt']

    table = Table(
        title='🏗️ Backend Modules',
        show_header=True, header_style='bold blue'
    )
    table.add_column('File', style='cyan', min_width=24)
    table.add_column('Size', justify='right')
    table.add_column('Status', justify='center')

    all_ok = True
    for filename in expected_files:
        fpath = backend_dir / filename
        if fpath.exists():
            size_kb = fpath.stat().st_size / 1024
            table.add_row(
                filename,
                f'{size_kb:.1f} KB',
                '[bold green]✔[/bold green]'
            )
        else:
            table.add_row(filename, 'N/A', '[bold red]✖ MISSING[/bold red]')
            all_ok = False

    console.print(table)

    if all_ok:
        console.print('[bold green]✔ All backend modules generated successfully.[/bold green]')
    else:
        console.print('[bold red]✖ Some backend modules are missing![/bold red]')


verify_backend_modules(BACKEND_DIR)

---
## 18. ✅ Final Inference & Deployment Summary
Display a comprehensive summary of the entire inference pipeline.

In [ ]:
def display_final_summary(
    batch_results: Dict[str, Any],
    model_path: Path,
    inference_dir: Path,
    backend_dir: Path
) -> None:
    """Display the final pipeline summary."""
    # Count generated files
    json_count = len(list((inference_dir / 'json').glob('*.json'))) if (inference_dir / 'json').exists() else 0
    img_count = len(list((inference_dir / 'images').glob('*.*'))) if (inference_dir / 'images').exists() else 0
    csv_count = len(list((inference_dir / 'csv').glob('*.csv'))) if (inference_dir / 'csv').exists() else 0
    backend_count = len(list(backend_dir.glob('*.*'))) if backend_dir.exists() else 0

    # Extract batch stats
    total_images = batch_results.get('processed_images', batch_results.get('total_images', 0))
    total_objects = batch_results.get('total_plastic_objects', 0)
    avg_conf = batch_results.get('average_confidence', 0.0)
    avg_time = batch_results.get('average_inference_time_ms', 0.0)
    avg_fps = batch_results.get('average_fps', 0.0)

    summary_text = (
        f'[bold green]✔ Model Loaded[/bold green]              {model_path.name}\n'
        f'[bold green]✔ Images Processed[/bold green]          {total_images}\n'
        f'[bold green]✔ Total Plastic Objects[/bold green]     {total_objects}\n'
        f'[bold green]✔ Average Confidence[/bold green]        {avg_conf:.4f}\n'
        f'[bold green]✔ Average Inference Time[/bold green]    {avg_time:.2f} ms\n'
        f'[bold green]✔ Average FPS[/bold green]               {avg_fps:.2f}\n'
        f'[bold green]✔ Annotated Images[/bold green]          {img_count}\n'
        f'[bold green]✔ JSON Files Generated[/bold green]      {json_count}\n'
        f'[bold green]✔ CSV Files Generated[/bold green]       {csv_count}\n'
        f'[bold green]✔ Backend Modules Generated[/bold green] {backend_count}\n'
        f'[bold green]✔ Ready for FastAPI Integration[/bold green]\n'
        f'\n'
        f'[bold cyan]─── Output Locations ───[/bold cyan]\n'
        f'  📁 Annotated Images:  {inference_dir / "images"}\n'
        f'  📁 JSON Results:      {inference_dir / "json"}\n'
        f'  📁 CSV Results:       {inference_dir / "csv"}\n'
        f'  📁 Reports:           {inference_dir / "reports"}\n'
        f'  📁 Backend Modules:   {backend_dir}\n'
        f'\n'
        f'[bold red]Next Notebook:[/bold red]  09_Severity_Engine.ipynb\n'
        f'The inference pipeline created in this notebook will be used by\n'
        f'the FastAPI backend and frontend for real-time plastic detection.'
    )

    console.print(Panel.fit(
        summary_text,
        title='🏁 PlasticSense AI — Inference & Deployment Complete',
        border_style='bold green'
    ))


display_final_summary(batch_results, BEST_PT_PATH, INFERENCE_DIR, BACKEND_DIR)
logger.info('=== INFERENCE & DEPLOYMENT PIPELINE COMPLETE ===')

## Export to ONNX\nExport the trained PyTorch model to ONNX format for production deployment.

In [ ]:
from ultralytics import YOLO\n\n# Load the best model\nmodel = YOLO(PROJECT_ROOT / 'runs' / 'detect' / 'train' / 'weights' / 'best.pt')\n\n# Export to ONNX\nsuccess = model.export(format='onnx', opset=12, dynamic=True)\nif success:\n    print('ONNX Export Successful!')